In [1]:
from granuscore.loader import JSONLineReader
from training_scripts.config import PROJECT_DIR
import pandas as pd

reader = JSONLineReader()

In [2]:
def abbreviate(name):
    repl = {
        "scope_": "",
        "document": "doc",
        "sentence": "sent",
        "spooling_": "",
        "pooling": "pool",
        "doc_mean_": "doc_",
        "lower_quantile_mean": "lqm",
        "stailq": "stq",
        "tailq_": "",
    }

    for k, v in repl.items():
        name = name.replace(k, v)

    name = name.replace("_", "-")
    if "stq-" in name:
        stq = name.split("stq-")[1].split("-")[0]
        name = name.replace("sent-lqm-", f"sent-lqm-{stq}-")
        name = name.replace(f"-stq-{stq}", "")
    return name

stats = []

data = reader.read(f'{PROJECT_DIR}/data/s2orc/evaluations.jsonl')

for d in data:
    name = d['name']
    stats.append({
        'name': abbreviate(name),
        'correct_ordering': d['correct_ordering'],
    })
paper_stats = pd.DataFrame(stats)

In [3]:
paper_stats

,name,correct_ordering
0,sent-weighted-mean-pool-sum,0.584867
1,sent-weighted-mean-pool-mean,0.664622
2,sent-weighted-mean-pool-lqm-0.1,0.626789
3,sent-weighted-mean-pool-lqm-0.3,0.668712
4,sent-weighted-mean-pool-lqm-0.5,0.672802
...,...,...
58,doc-pool-lqm-0.1,0.648262
59,doc-pool-lqm-0.3,0.656442
60,doc-pool-lqm-0.5,0.660532
61,doc-pool-min,0.599182


In [4]:
def abbreviate(name):
    repl = {
        "scope_": "",
        "document": "doc",
        "sentence": "sent",
        "spooling_": "",
        "pooling": "pool",
        "doc_mean_": "doc_",
        "lower_quantile_mean": "lqm",
        "stailq": "stq",
        "tailq_": "",
    }

    for k, v in repl.items():
        name = name.replace(k, v)

    name = name.replace("_", "-")
    if "stq-" in name:
        stq = name.split("stq-")[1].split("-")[0]
        name = name.replace("sent-lqm-", f"sent-lqm-{stq}-")
        name = name.replace(f"-stq-{stq}", "")
    return name

stats = []

evaluations = {
    'news': reader.read(f"{PROJECT_DIR}/data/news_articles/evaluation/stats-news-articles.jsonl"),
    'movie': reader.read(f"{PROJECT_DIR}/data/domain-agnostic/evaluation/movie-stats-domain-agnostic.jsonl"),
    'twitter': reader.read(f"{PROJECT_DIR}/data/domain-agnostic/evaluation/twitter-stats-domain-agnostic.jsonl"),
    'yelp': reader.read(f"{PROJECT_DIR}/data/domain-agnostic/evaluation/yelp-stats-domain-agnostic.jsonl")
}

sizes = {
    'movie': 920,
    'twitter': 984,
    'yelp': 845,
    'news': 573
}
total = sum(sizes.values())


for i in range(len(evaluations['news'])):
    n = evaluations['news'][i]
    t = evaluations['twitter'][i]
    y = evaluations['yelp'][i]
    m = evaluations['movie'][i]
    assert n['model'] == t['model'] == y['model'] == m['model']

    explained_deviance = 0
    for domain, size in sizes.items():
        if domain == 'movie':
            value = m['explained_deviance']['length_plus_granularity'] - m['explained_deviance']['length_only']
        elif domain == 'twitter':
            value = t['explained_deviance']['length_plus_granularity'] - t['explained_deviance']['length_only']
        elif domain == 'yelp':
            value = y['explained_deviance']['length_plus_granularity'] - y['explained_deviance']['length_only']
        elif domain == 'news':
            value = n['explained_deviance']['length_plus_granularity'] - n['explained_deviance']['length_only']
        else:
            raise ValueError
        explained_deviance += value * size
    explained_deviance = explained_deviance / total

    name = n['model']
    stats.append({
        'name': abbreviate(name),
        'explained_deviance': explained_deviance,
    })
sentence_specificity_stats = pd.DataFrame(stats)

In [5]:
sentence_specificity_stats

,name,explained_deviance
0,sent-sum-pool-sum,0.036724
1,sent-sum-pool-mean,0.067523
2,sent-sum-pool-lqm-0.1,0.066448
3,sent-sum-pool-lqm-0.3,0.072708
4,sent-sum-pool-lqm-0.5,0.071195
...,...,...
58,sent-weighted-mean-pool-lqm-0.1,0.091379
59,sent-weighted-mean-pool-lqm-0.3,0.094074
60,sent-weighted-mean-pool-lqm-0.5,0.095412
61,sent-weighted-mean-pool-min,0.086851


In [6]:
from scipy.stats import pearsonr

df1_sorted = sentence_specificity_stats.sort_values("explained_deviance", ascending=True).reset_index(drop=True)
df2_sorted = paper_stats.sort_values("correct_ordering", ascending=True).reset_index(drop=True)

# convert ordering to rank based on sorted order
rank1 = {name: i for i, name in enumerate(df1_sorted["name"])}
rank2 = {name: i for i, name in enumerate(df2_sorted["name"])}

# keep only common names
common = set(rank1) & set(rank2)

r1 = [rank1[n] for n in common]
r2 = [rank2[n] for n in common]

pearson, p = pearsonr(r1, r2)

print("Pearson r:", pearson)
print("p-value:", p)

Pearson r: 0.6241839477726576
p-value: 4.6034838727622275e-08
